# Types of Mechanisms for Subgraph
1) You create two seperate graphs, and in one of the nodes of parent graph, call invoke method of another
child graph.(both have seperate states)
2) subgraph directly added as a node in the parent (shared state)

In [36]:
from langchain_groq import ChatGroq
from langgraph.graph import START,END,StateGraph
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,BaseMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import add_messages
from dotenv import load_dotenv
from typing import TypedDict,Annotated,Dict,Sequence,List

In [37]:
class substate(TypedDict):
    inp:str
    trans:str

class parstate(TypedDict):
    inp:str
    anseng:str
    anshin:str

In [38]:
tllm=ChatGroq(model='llama-3.1-8b-instant')
pllm=ChatGroq(model='llama-3.1-8b-instant')

In [39]:
def translate(state:substate)->substate:
    prompt=f'''YOu are a helpful assistant. Translate the input to Hindi, Keep it clear, don't change content
        Text: {state['inp']}
    '''
    res=tllm.invoke(prompt).content
    return {'trans':res}

def answer(state:parstate)->parstate:
    prompt=f''' As a helpful assistant, answer the given question in brief:
     question: {state['inp']} '''
    res=pllm.invoke(prompt).content
    return {'anseng':res}

In [40]:
subgraph=StateGraph(substate)
subgraph.add_node("translate",translate)
subgraph.add_edge(START,"translate")
subgraph.add_edge("translate",END)

subapp=subgraph.compile()

In [41]:
def translator(state:parstate):
    print("received actual is: ",state['inp'])
    result=subapp.invoke({'inp':{state['anseng']}})
    return {'anshin':result['trans']}

In [42]:
parentgraph=StateGraph(parstate)
parentgraph.add_node("generator",answer)
parentgraph.add_edge(START,"generator")
parentgraph.add_node("Translator",translator)
parentgraph.add_edge("generator","Translator")
parentgraph.add_edge("Translator",END)

parapp=parentgraph.compile()

In [43]:
res=parapp.invoke({'inp':"Tell me in brief about India"})
print(res['inp'])
print(res['anseng'])
print(res['anshin'])

received actual is:  Tell me in brief about India
Tell me in brief about India
India is a country with a rich history and culture. Here are some key points about India:

- **Location**: South Asia, bordered by Pakistan, China, Nepal, Bhutan, Bangladesh, and Myanmar.
- **Population**: Over 1.38 billion people, making it the second-most populous country.
- **Language**: 22 officially recognized languages, with Hindi and English being the most widely spoken.
- **Cuisine**: Diverse and flavorful, with popular dishes like curries, naan bread, and tandoori chicken.
- **Religion**: Mainly Hindu, with significant Muslim and Christian populations.
- **Economy**: The world's fifth-largest economy, driven by IT, manufacturing, and services.

India is a vibrant and diverse country with a unique blend of history, culture, and modernity.
भारत एक इतिहास और संस्कृति से भरा देश है। यहाँ भारत के बारे में कुछ महत्वपूर्ण बिन्दु हैं:

- **स्थान**: दक्षिण एशिया, पाकिस्तान, चीन, नेपाल, भूटान, बांग्लादेश, और 